# Comparison with pyvinecopulib

This notebook compares `pyscarcopula` against `pyvinecopulib` in MLE-only settings on `data/sp500_yfinance_2020_prices.csv`.

Comparison directions:

- bivariate Gaussian copula fit, likelihood, density evaluation, and sampling;
- d-dimensional R-vine fit, likelihood evaluation, and sampling for `d = 10, 50, 100`;
- detailed sampling and conditional sampling diagnostics for `d = 10`;
- selected vine structure comparison.

The default setup uses Gaussian pair-copulas and a shallow truncation level. This keeps the focus on computational accuracy/performance and makes the `d=100` run practical. Remove or change these controls if you want a full family-selection benchmark.

In [42]:
import inspect
import time
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats

try:
    from IPython.display import display
except ImportError:
    display = print

import pyvinecopulib as pv

from pyscarcopula._utils import pobs
from pyscarcopula import GumbelCopula, RVineCopula, PredictConfig


In [43]:
DATA_DIR = Path('data')
if not DATA_DIR.exists():
    DATA_DIR = Path('..') / 'data'

DATA_FILE = DATA_DIR / 'sp500_yfinance_2020_prices.csv'
DIMS = [10, 50, 100]
TRUNCATION_LEVEL = 2
N_SAMPLE_BY_DIM = {10: 5000, 50: 2000, 100: 1000}
N_CONDITIONAL = 3000
MMD_SUBSAMPLE = 600
COND_MCMC_STEPS = 180
COND_MCMC_BURNIN = 80
SEED = 2026

COPULA_LABEL = 'GumbelCopula()'
PS_BICOP_CLASS = GumbelCopula
PS_BICOP_ROTATION = 0
PS_CANDIDATES = [GumbelCopula]
PS_ALLOW_ROTATIONS = False
PS_FIXED_COPULA_ROTATION = 0

PV_FAMILY = pv.BicopFamily.gumbel
PV_FAMILY_SET = [PV_FAMILY]
PV_FIXED_ROTATION = 0
PV_STRUCTURE_ALLOW_ROTATIONS = False


## Statistical tests used here

For **conditional sampling with 9 of 10 variables fixed**, the free coordinate is one-dimensional. This is the cleanest comparison and the notebook reports:

- two-sample Kolmogorov-Smirnov statistic and p-value;
- two-sample Cramer-von Mises statistic and p-value;
- Wasserstein distance;
- mean, standard deviation, and quantile differences.

For **unconditional multivariate sampling**, there is no single perfect scalar test. The notebook uses complementary diagnostics:

- per-coordinate two-sample KS tests between samples;
- correlation-matrix Frobenius distance;
- an RBF-MMD estimate on a subsample.

For **structure**, the notebook compares variable order and tree-0 edge overlap. Exact matrix equality is not expected when the two libraries run independent structure selection, even with the same pair family.

In [44]:
prices = pd.read_csv(DATA_FILE, parse_dates=['Date']).set_index('Date')
prices = prices.apply(pd.to_numeric, errors='coerce')
prices = prices.dropna(axis=1, thresh=int(0.95 * len(prices))).ffill().dropna()
returns = np.log(prices / prices.shift(1)).dropna()

assets = list(returns.columns)
available_dims = [d for d in DIMS if d <= len(assets)]
unavailable_dims = [d for d in DIMS if d > len(assets)]

print(f'Prices: {prices.shape[0]} rows, {prices.shape[1]} assets')
print(f'Returns: {returns.shape[0]} rows, {returns.shape[1]} assets')
print('Available benchmark dimensions:', available_dims)
if unavailable_dims:
    print('Skipped dimensions, not enough assets:', unavailable_dims)


def dataset_for_dim(d):
    cols = assets[:d]
    u = pobs(returns[cols].values)
    return np.asarray(u, dtype=np.float64), cols


Prices: 1582 rows, 105 assets
Returns: 1581 rows, 105 assets
Available benchmark dimensions: [10, 50, 100]


In [45]:
def as_fortran(u):
    return np.asarray(u, dtype=np.float64, order='F')


def timed(label, fn):
    print(f'start {label}', flush=True)
    t0 = time.perf_counter()
    out = fn()
    elapsed = time.perf_counter() - t0
    print(f'done  {label}: {elapsed:.3f}s', flush=True)
    return out, elapsed


def make_ps_bicop():
    return PS_BICOP_CLASS(rotate=PS_BICOP_ROTATION)


def fixed_ps_copula_specs(d):
    if PS_FIXED_COPULA_ROTATION is None:
        return None
    return [
        [(PS_CANDIDATES[0], PS_FIXED_COPULA_ROTATION) for _ in range(d - 1 - tree)]
        for tree in range(d - 1)
    ]


def fixed_pv_pair_copulas(d):
    if PV_FIXED_ROTATION is None:
        return []
    n_levels = d - 1 if TRUNCATION_LEVEL is None else min(TRUNCATION_LEVEL, d - 1)
    return [
        [pv.Bicop(PV_FAMILY, PV_FIXED_ROTATION) for _ in range(d - 1 - tree)]
        for tree in range(n_levels)
    ]


def pv_vine_controls():
    return pv.FitControlsVinecop(
        family_set=PV_FAMILY_SET,
        allow_rotations=PV_STRUCTURE_ALLOW_ROTATIONS,
        trunc_lvl=TRUNCATION_LEVEL,
        num_threads=1,
        preselect_families=False,
    )


def pv_bicop_controls():
    return pv.FitControlsBicop(
        family_set=PV_FAMILY_SET,
        allow_rotations=PV_STRUCTURE_ALLOW_ROTATIONS,
        preselect_families=False,
    )


def fit_ps_vine(u):
    return RVineCopula(
        candidates=PS_CANDIDATES,
        allow_rotations=PS_ALLOW_ROTATIONS,
    ).fit(
        u,
        method='mle',
        truncation_level=TRUNCATION_LEVEL,
        copulas=fixed_ps_copula_specs(u.shape[1]),
    )


def fit_pv_bicop(u):
    bicop = pv.Bicop(PV_FAMILY, PV_FIXED_ROTATION)
    bicop.fit(as_fortran(u), controls=pv.FitControlsBicop())
    return bicop


def fit_pv_vine(u):
    data = as_fortran(u)
    selected = pv.Vinecop.from_data(data, controls=pv_vine_controls())
    if PV_FIXED_ROTATION is None:
        return selected
    model = pv.Vinecop.from_structure(
        structure=selected.structure,
        pair_copulas=fixed_pv_pair_copulas(u.shape[1]),
    )
    model.fit(data, controls=pv.FitControlsBicop(), num_threads=1)
    return model


def corr_frobenius(a, b):
    ca = np.corrcoef(a, rowvar=False)
    cb = np.corrcoef(b, rowvar=False)
    return float(np.linalg.norm(ca - cb, ord='fro') / ca.shape[0])


def marginal_ks_summary(x, y):
    rows = []
    for j in range(x.shape[1]):
        res = stats.ks_2samp(x[:, j], y[:, j])
        rows.append({'dim': j, 'ks_stat': res.statistic, 'ks_pvalue': res.pvalue})
    frame = pd.DataFrame(rows)
    return {
        'ks_stat_mean': float(frame['ks_stat'].mean()),
        'ks_stat_max': float(frame['ks_stat'].max()),
        'ks_pvalue_min': float(frame['ks_pvalue'].min()),
    }


def rbf_mmd2(x, y, max_n=MMD_SUBSAMPLE, seed=SEED):
    rng = np.random.default_rng(seed)
    if len(x) > max_n:
        x = x[rng.choice(len(x), size=max_n, replace=False)]
    if len(y) > max_n:
        y = y[rng.choice(len(y), size=max_n, replace=False)]
    z = np.vstack([x, y])
    sq = np.sum((z[:, None, :] - z[None, :, :]) ** 2, axis=2)
    positive = sq[sq > 0]
    gamma = 1.0 / np.median(positive) if positive.size else 1.0
    kxx = np.exp(-gamma * np.sum((x[:, None, :] - x[None, :, :]) ** 2, axis=2))
    kyy = np.exp(-gamma * np.sum((y[:, None, :] - y[None, :, :]) ** 2, axis=2))
    kxy = np.exp(-gamma * np.sum((x[:, None, :] - y[None, :, :]) ** 2, axis=2))
    return float(kxx.mean() + kyy.mean() - 2.0 * kxy.mean())


def _rbf_mmd2_from_kernel(kernel, idx_x, idx_y):
    kxx = kernel[np.ix_(idx_x, idx_x)]
    kyy = kernel[np.ix_(idx_y, idx_y)]
    kxy = kernel[np.ix_(idx_x, idx_y)]
    return float(kxx.mean() + kyy.mean() - 2.0 * kxy.mean())


def rbf_mmd2_permutation_test(x, y, max_n=MMD_SUBSAMPLE, n_perm=199, seed=SEED):
    rng = np.random.default_rng(seed)
    x = np.asarray(x, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)
    n = min(len(x), len(y), max_n)
    if n < 2:
        return {'mmd2': np.nan, 'pvalue': np.nan, 'n': n, 'n_perm': n_perm}
    if len(x) > n:
        x = x[rng.choice(len(x), size=n, replace=False)]
    if len(y) > n:
        y = y[rng.choice(len(y), size=n, replace=False)]

    z = np.vstack([x, y])
    sq = np.sum((z[:, None, :] - z[None, :, :]) ** 2, axis=2)
    positive = sq[sq > 0]
    gamma = 1.0 / np.median(positive) if positive.size else 1.0
    kernel = np.exp(-gamma * sq)

    idx_x = np.arange(n)
    idx_y = np.arange(n, 2 * n)
    observed = _rbf_mmd2_from_kernel(kernel, idx_x, idx_y)

    count = 0
    base = np.arange(2 * n)
    for _ in range(n_perm):
        perm = rng.permutation(base)
        stat = _rbf_mmd2_from_kernel(kernel, perm[:n], perm[n:])
        if stat >= observed:
            count += 1
    pvalue = (count + 1.0) / (n_perm + 1.0)
    return {'mmd2': observed, 'pvalue': pvalue, 'n': n, 'n_perm': n_perm}


def pairwise_sampling_diagnostics(samples, max_n=MMD_SUBSAMPLE, seed=SEED):
    rows = []
    for offset, (left_name, right_name, left, right) in enumerate(samples):
        ks = marginal_ks_summary(left, right)
        mmd = rbf_mmd2_permutation_test(
            left, right, max_n=max_n, seed=seed + 1009 * offset)
        rows.append({
            'comparison': f'{left_name} vs {right_name}',
            'n_left': len(left),
            'n_right': len(right),
            'marginal_ks_stat_mean': ks['ks_stat_mean'],
            'marginal_ks_stat_max': ks['ks_stat_max'],
            'marginal_ks_pvalue_min': ks['ks_pvalue_min'],
            'corr_fro': corr_frobenius(left, right),
            'rbf_mmd2': mmd['mmd2'],
            'rbf_mmd_pvalue': mmd['pvalue'],
            'mmd_subsample_n': mmd['n'],
            'mmd_permutations': mmd['n_perm'],
        })
    return pd.DataFrame(rows)


def ps_order(vine):
    return [int(vine.matrix[vine.d - 1 - col, col]) for col in range(vine.d)]


def pv_order(vine):
    # pyvinecopulib stores structure variables as 1-based ids.
    return [int(x) - 1 for x in vine.structure.order]


def ps_tree_edges(vine, tree):
    return {
        (tuple(sorted(int(v) for v in conditioned)), tuple(sorted(int(v) for v in conditioning)))
        for conditioned, conditioning in vine.trees[tree]
    }


def ps_tree0_edges(vine):
    return {conditioned for conditioned, _ in ps_tree_edges(vine, 0)}


def pv_tree_edges(vine, tree):
    # For truncated pyvinecopulib models, vine.matrix contains zeros outside
    # active entries. Decode through RVineStructure instead: order[e] is one
    # conditioned endpoint, struct_array(tree, e, False) is the other, and
    # lower tree struct_array entries are the conditioning set. All are 1-based.
    order = pv_order(vine)
    edges = set()
    for edge_idx in range(vine.dim - 1 - tree):
        conditioned = tuple(sorted((
            int(vine.structure.struct_array(tree, edge_idx, False)) - 1,
            order[edge_idx],
        )))
        conditioning = tuple(sorted(
            int(vine.structure.struct_array(k, edge_idx, False)) - 1
            for k in range(tree)
        ))
        edges.add((conditioned, conditioning))
    return edges


def pv_tree0_edges(vine):
    return {conditioned for conditioned, _ in pv_tree_edges(vine, 0)}


def structure_edge_label(edge, cols):
    conditioned, conditioning = edge
    label = f'{cols[conditioned[0]]} - {cols[conditioned[1]]}'
    if conditioning:
        label += ' | ' + ', '.join(cols[i] for i in conditioning)
    return label


def package_source_check():
    import pyscarcopula
    import pyscarcopula.vine._rvine_dissmann as rvine_dissmann
    source = inspect.getsource(rvine_dissmann._score_candidate_structure)
    return {
        'pyscarcopula': str(Path(inspect.getfile(pyscarcopula)).resolve()),
        'rvine_dissmann': str(Path(inspect.getfile(rvine_dissmann)).resolve()),
        'has_empty_given_fast_path': 'if given_vars:' in source,
    }


## Warm up

Run this cell before the timed sections. It primes first-call overhead from Python, NumPy/SciPy, the copula fitting paths, sampling paths, and the conditional-sampling paths without including that cost in the benchmark tables below.

In [46]:
RUN_WARMUP = True
WARMUP_DIM = min(10, max(available_dims))

def run_warmup():
    u_warm, _ = dataset_for_dim(WARMUP_DIM)
    warmup_n = min(250, len(u_warm))
    u_fit = u_warm[:warmup_n]
    u_bi = u_warm[:, :2]
    probe_bi = np.clip(u_bi[:64], 1e-6, 1.0 - 1e-6)

    ps_bi = make_ps_bicop()
    ps_bi_result = ps_bi.fit(u_bi, method='mle')
    pv_bi = fit_pv_bicop(u_bi)
    _ = np.exp(ps_bi.log_pdf(
        probe_bi[:, 0],
        probe_bi[:, 1],
        np.full(len(probe_bi), ps_bi_result.copula_param),
    ))
    _ = pv_bi.pdf(as_fortran(probe_bi))
    _ = ps_bi.sample(64, r=ps_bi_result.copula_param, rng=np.random.default_rng(SEED - 1))
    _ = pv_bi.simulate(64, seeds=[SEED - 1])

    ps_vine = fit_ps_vine(u_fit)
    pv_vine = fit_pv_vine(u_fit)
    _ = ps_vine.log_likelihood(u_fit)
    _ = pv_vine.loglik(as_fortran(u_fit))
    _ = ps_vine.sample(64, rng=np.random.default_rng(SEED - 2))
    _ = pv_vine.simulate(64, seeds=[SEED - 2])

    if WARMUP_DIM >= 10:
        x0 = u_fit[-1].copy()
        free_var = pv_order(pv_vine)[0]
        given = {j: float(x0[j]) for j in range(WARMUP_DIM) if j != free_var}
        given_vars = list(given)

        z0 = pv_vine.rosenblatt(as_fortran(x0.reshape(1, -1)), randomize_discrete=False)
        rng = np.random.default_rng(SEED - 3)
        z = rng.uniform(1e-10, 1.0 - 1e-10, size=(16, WARMUP_DIM))
        z[:, given_vars] = z0[0, given_vars]
        _ = pv_vine.inverse_rosenblatt(as_fortran(z))

        pcfg = PredictConfig(
            given=given,
            horizon='next',
            return_diagnostics=True,
            mcmc_steps=20,
            mcmc_burnin=10,
        )
        _ = ps_vine.predict(
            16,
            u_train=u_fit,
            predict_config=pcfg,
            rng=np.random.default_rng(SEED - 4),
        )

    return {
        'warmup_dim': WARMUP_DIM,
        'warmup_observations': warmup_n,
        'copula': COPULA_LABEL,
        'bivariate': 'fit/pdf/sample',
        'vine': 'fit/loglik/sample',
        'conditional': WARMUP_DIM >= 10,
    }

warmup_result = run_warmup() if RUN_WARMUP else {'skipped': True}
warmup_result


{'warmup_dim': 10,
 'warmup_observations': 250,
 'copula': 'GumbelCopula()',
 'bivariate': 'fit/pdf/sample',
 'vine': 'fit/loglik/sample',
 'conditional': True}

## Bivariate case

Both libraries fit the configured bivariate copula family to the same pseudo-observations.

In [47]:
u2, cols2 = dataset_for_dim(2)

ps_bi = make_ps_bicop()
ps_result, ps_fit_time = timed('pyscarcopula bicop fit', lambda: ps_bi.fit(u2, method='mle'))
pv_bi, pv_fit_time = timed('pyvinecopulib bicop fit', lambda: fit_pv_bicop(u2))

_, ps_pdf_time = timed(
    'pyscarcopula bicop pdf',
    lambda: np.exp(ps_bi.log_pdf(u2[:, 0], u2[:, 1], np.full(len(u2), ps_result.copula_param))),
)
pv_pdf, pv_pdf_time = timed('pyvinecopulib bicop pdf', lambda: pv_bi.pdf(as_fortran(u2)))
ps_pdf = np.exp(ps_bi.log_pdf(u2[:, 0], u2[:, 1], np.full(len(u2), ps_result.copula_param)))

ps_sample, ps_sample_time = timed(
    'pyscarcopula bicop sample',
    lambda: ps_bi.sample(5000, r=ps_result.copula_param, rng=np.random.default_rng(SEED)),
)
pv_sample, pv_sample_time = timed(
    'pyvinecopulib bicop sample',
    lambda: pv_bi.simulate(5000, seeds=[SEED]),
)

ps_param = float(ps_result.copula_param)
pv_params = np.asarray(pv_bi.parameters, dtype=float).ravel()
pv_param = float(pv_params[0]) if pv_params.size else 0.0

pd.DataFrame([{
    'assets': ', '.join(cols2),
    'copula': COPULA_LABEL,
    'param_pyscarcopula': ps_param,
    'param_pyvinecopulib': pv_param,
    'param_abs_diff': abs(ps_param - pv_param),
    'family_pyvinecopulib': str(pv_bi.family),
    'rotation_pyvinecopulib': int(pv_bi.rotation),
    'loglik_pyscarcopula': ps_result.log_likelihood,
    'loglik_pyvinecopulib': pv_bi.loglik(as_fortran(u2)),
    'pdf_max_abs_diff': float(np.max(np.abs(ps_pdf - pv_pdf))),
    'fit_time_pyscarcopula_s': ps_fit_time,
    'fit_time_pyvinecopulib_s': pv_fit_time,
    'pdf_time_pyscarcopula_s': ps_pdf_time,
    'pdf_time_pyvinecopulib_s': pv_pdf_time,
    'sample_time_pyscarcopula_s': ps_sample_time,
    'sample_time_pyvinecopulib_s': pv_sample_time,
    'sample_ks_dim0': stats.ks_2samp(ps_sample[:, 0], pv_sample[:, 0]).statistic,
    'sample_ks_dim1': stats.ks_2samp(ps_sample[:, 1], pv_sample[:, 1]).statistic,
}]).T


start pyscarcopula bicop fit
done  pyscarcopula bicop fit: 0.005s
start pyvinecopulib bicop fit
done  pyvinecopulib bicop fit: 0.004s
start pyscarcopula bicop pdf
done  pyscarcopula bicop pdf: 0.000s
start pyvinecopulib bicop pdf
done  pyvinecopulib bicop pdf: 0.001s
start pyscarcopula bicop sample
done  pyscarcopula bicop sample: 0.002s
start pyvinecopulib bicop sample
done  pyvinecopulib bicop sample: 0.002s


,0
assets,"AAPL, ABBV"
copula,GumbelCopula()
param_pyscarcopula,1.152249
param_pyvinecopulib,1.152249
param_abs_diff,0.0
family_pyvinecopulib,BicopFamily.gumbel
rotation_pyvinecopulib,0
loglik_pyscarcopula,44.986205
loglik_pyvinecopulib,44.986205
pdf_max_abs_diff,0.000026


## R-vine performance and sampling diagnostics

The loop below fits both libraries for each available target dimension. It stores the `d=10` models for the detailed structure, sampling, and conditional-sampling sections.

In [48]:
benchmark_rows = []
models = {}

for d in available_dims:
    u_d, cols_d = dataset_for_dim(d)
    n_sample = N_SAMPLE_BY_DIM.get(d, 1000)

    ps_vine, ps_fit_t = timed(f'pyscarcopula fit d={d}', lambda u=u_d: fit_ps_vine(u))
    pv_vine, pv_fit_t = timed(f'pyvinecopulib fit d={d}', lambda u=u_d: fit_pv_vine(u))

    ps_ll, ps_ll_t = timed('pyscarcopula loglik', lambda: ps_vine.log_likelihood(u_d))
    pv_ll, pv_ll_t = timed('pyvinecopulib loglik', lambda: pv_vine.loglik(as_fortran(u_d)))

    ps_samp, ps_sample_t = timed(
        'pyscarcopula sample',
        lambda: ps_vine.sample(n_sample, rng=np.random.default_rng(SEED + d)),
    )
    pv_samp, pv_sample_t = timed(
        'pyvinecopulib sample',
        lambda: pv_vine.simulate(n_sample, seeds=[SEED + d]),
    )

    ps_ks = marginal_ks_summary(ps_samp, u_d)
    pv_ks = marginal_ks_summary(pv_samp, u_d)

    benchmark_rows.append({
        'd': d,
        'n_obs': len(u_d),
        'n_sample': n_sample,
        'fit_time_pyscarcopula_s': ps_fit_t,
        'fit_time_pyvinecopulib_s': pv_fit_t,
        'loglik_pyscarcopula': ps_ll,
        'loglik_pyvinecopulib': pv_ll,
        'loglik_diff_per_obs': (ps_ll - pv_ll) / len(u_d),
        'loglik_time_pyscarcopula_s': ps_ll_t,
        'loglik_time_pyvinecopulib_s': pv_ll_t,
        'sample_time_pyscarcopula_s': ps_sample_t,
        'sample_time_pyvinecopulib_s': pv_sample_t,
        'ps_sample_corr_fro_to_data': corr_frobenius(ps_samp, u_d),
        'pv_sample_corr_fro_to_data': corr_frobenius(pv_samp, u_d),
        'ps_sample_ks_stat_mean': ps_ks['ks_stat_mean'],
        'pv_sample_ks_stat_mean': pv_ks['ks_stat_mean'],
        'ps_sample_ks_stat_max': ps_ks['ks_stat_max'],
        'pv_sample_ks_stat_max': pv_ks['ks_stat_max'],
    })
    models[d] = {
        'u': u_d,
        'cols': cols_d,
        'ps_vine': ps_vine,
        'pv_vine': pv_vine,
        'ps_sample': ps_samp,
        'pv_sample': pv_samp,
    }

benchmark = pd.DataFrame(benchmark_rows)
benchmark


start pyscarcopula fit d=10
done  pyscarcopula fit d=10: 0.098s
start pyvinecopulib fit d=10
done  pyvinecopulib fit d=10: 0.182s
start pyscarcopula loglik
done  pyscarcopula loglik: 0.009s
start pyvinecopulib loglik
done  pyvinecopulib loglik: 0.014s
start pyscarcopula sample
done  pyscarcopula sample: 0.035s
start pyvinecopulib sample
done  pyvinecopulib sample: 0.041s
start pyscarcopula fit d=50
done  pyscarcopula fit d=50: 0.876s
start pyvinecopulib fit d=50
done  pyvinecopulib fit d=50: 1.476s
start pyscarcopula loglik
done  pyscarcopula loglik: 0.046s
start pyvinecopulib loglik
done  pyvinecopulib loglik: 0.070s
start pyscarcopula sample
done  pyscarcopula sample: 0.086s
start pyvinecopulib sample
done  pyvinecopulib sample: 0.090s
start pyscarcopula fit d=100
done  pyscarcopula fit d=100: 2.951s
start pyvinecopulib fit d=100
done  pyvinecopulib fit d=100: 4.073s
start pyscarcopula loglik
done  pyscarcopula loglik: 0.093s
start pyvinecopulib loglik
done  pyvinecopulib loglik: 0.1

,d,n_obs,n_sample,fit_time_pyscarcopula_s,fit_time_pyvinecopulib_s,loglik_pyscarcopula,loglik_pyvinecopulib,loglik_diff_per_obs,loglik_time_pyscarcopula_s,loglik_time_pyvinecopulib_s,sample_time_pyscarcopula_s,sample_time_pyvinecopulib_s,ps_sample_corr_fro_to_data,pv_sample_corr_fro_to_data,ps_sample_ks_stat_mean,pv_sample_ks_stat_mean,ps_sample_ks_stat_max,pv_sample_ks_stat_max
0,10,1581,5000,0.098395,0.181803,3316.065595,3316.065585,6.418184e-09,0.008984,0.013593,0.034841,0.041299,0.092303,0.095716,0.011622,0.012963,0.016085,0.017584
1,50,1581,2000,0.875776,1.476430,26278.554060,26277.801713,4.758682e-04,0.046302,0.069526,0.085943,0.090159,0.210239,0.208214,0.018651,0.020986,0.031172,0.038412
2,100,1581,1000,2.950651,4.073480,55664.331179,55661.176519,1.995357e-03,0.092711,0.141990,0.106004,0.096333,0.231096,0.205272,0.026151,0.027540,0.054391,0.054390


## Vine structure comparison (`d=10`)

The two libraries use different matrix conventions. For `pyvinecopulib`, the notebook decodes edges through `RVineStructure.order` and `RVineStructure.struct_array(..., natural_order=False)`, which are 1-based. This avoids reading inactive zeros from the truncated `vine.matrix`.

In [49]:
if 10 not in models:
    raise ValueError('d=10 is required for the detailed structure comparison')

m10 = models[10]
ps_v10 = m10['ps_vine']
pv_v10 = m10['pv_vine']
cols10 = m10['cols']

ps_ord = ps_order(ps_v10)
pv_ord = pv_order(pv_v10)
active_structure_levels = min(TRUNCATION_LEVEL if TRUNCATION_LEVEL is not None else 9, 9)

structure_rows = []
for tree in range(active_structure_levels):
    ps_edges_t = ps_tree_edges(ps_v10, tree)
    pv_edges_t = pv_tree_edges(pv_v10, tree)
    union = ps_edges_t | pv_edges_t
    structure_rows.append({
        'tree': tree,
        'edges_pyscarcopula': len(ps_edges_t),
        'edges_pyvinecopulib': len(pv_edges_t),
        'common_edges': len(ps_edges_t & pv_edges_t),
        'jaccard': len(ps_edges_t & pv_edges_t) / len(union) if union else 1.0,
    })

structure_summary = pd.DataFrame([{
    'ps_order': [cols10[i] for i in ps_ord],
    'pyvinecopulib_order': [cols10[i] for i in pv_ord],
    'same_order': ps_ord == pv_ord,
    'active_tree_levels': active_structure_levels,
}]).T

display(structure_summary)
pd.DataFrame(structure_rows)


,0
ps_order,"[ABBV, AMD, AMAT, ADI, AAPL, ADBE, ACN, ADP, A..."
pyvinecopulib_order,"[AMD, AMAT, ADI, AAPL, ADBE, ACN, ADP, ABBV, A..."
same_order,False
active_tree_levels,2


,tree,edges_pyscarcopula,edges_pyvinecopulib,common_edges,jaccard
0,0,9,9,9,1.0
1,1,8,8,8,1.0


In [50]:
edge_rows = []
for tree in range(active_structure_levels):
    ps_edges_t = ps_tree_edges(ps_v10, tree)
    pv_edges_t = pv_tree_edges(pv_v10, tree)
    for edge in sorted(ps_edges_t | pv_edges_t):
        edge_rows.append({
            'tree': tree,
            'edge': structure_edge_label(edge, cols10),
            'in_pyscarcopula': edge in ps_edges_t,
            'in_pyvinecopulib': edge in pv_edges_t,
        })
pd.DataFrame(edge_rows)


,tree,edge,in_pyscarcopula,in_pyvinecopulib
0,0,AAPL - ADBE,True,True
1,0,AAPL - ADI,True,True
2,0,ABBV - AMGN,True,True
3,0,ABT - ADP,True,True
4,0,ABT - AMGN,True,True
5,0,ACN - ADBE,True,True
6,0,ACN - ADP,True,True
7,0,ADI - AMAT,True,True
8,0,AMAT - AMD,True,True
9,1,AAPL - ACN | ADBE,True,True


## Detailed unconditional sampling comparison (`d=10`)

This compares the two fitted models through their simulated samples and the empirical pseudo-observations. Samples are not expected to match row-by-row; the diagnostics look for distribution-level disagreement. The MMD p-value is a permutation diagnostic on a fixed subsample, so use it as a clear reject/no-reject signal rather than as an exact analytical test.

In [51]:
u10 = m10['u']

# Use an independent seed for the detailed d=10 sampling diagnostics so a
# single unlucky benchmark seed does not dominate the visual comparison.
DETAIL_SAMPLE_SEED = SEED + 100_000
DETAIL_SAMPLE_N = N_SAMPLE_BY_DIM.get(10, len(m10['ps_sample']))

ps_s10 = ps_v10.sample(DETAIL_SAMPLE_N, rng=np.random.default_rng(DETAIL_SAMPLE_SEED))
pv_s10 = pv_v10.simulate(DETAIL_SAMPLE_N, seeds=[DETAIL_SAMPLE_SEED])

sampling_summary = pairwise_sampling_diagnostics([
    ('pyscarcopula sample', 'pyvinecopulib sample', ps_s10, pv_s10),
    ('pyscarcopula sample', 'empirical data', ps_s10, u10),
    ('pyvinecopulib sample', 'empirical data', pv_s10, u10),
])
sampling_summary


,comparison,n_left,n_right,marginal_ks_stat_mean,marginal_ks_stat_max,marginal_ks_pvalue_min,corr_fro,rbf_mmd2,rbf_mmd_pvalue,mmd_subsample_n,mmd_permutations
0,pyscarcopula sample vs pyvinecopulib sample,5000,5000,0.018600,0.027800,0.041954,0.019841,0.001825,0.540,600,199
1,pyscarcopula sample vs empirical data,5000,1581,0.011996,0.016227,0.903461,0.092644,0.002944,0.100,600,199
2,pyvinecopulib sample vs empirical data,5000,1581,0.012506,0.016458,0.894250,0.099776,0.003445,0.055,600,199


In [52]:
ks_rows = []
for left_name, right_name, left, right in [
    ('pyscarcopula sample', 'pyvinecopulib sample', ps_s10, pv_s10),
    ('pyscarcopula sample', 'empirical data', ps_s10, u10),
    ('pyvinecopulib sample', 'empirical data', pv_s10, u10),
]:
    for j, name in enumerate(cols10):
        res = stats.ks_2samp(left[:, j], right[:, j])
        ks_rows.append({
            'comparison': f'{left_name} vs {right_name}',
            'asset': name,
            'ks_stat': res.statistic,
            'ks_pvalue': res.pvalue,
        })
ks_frame = pd.DataFrame(ks_rows)
ks_frame.sort_values(['comparison', 'ks_stat'], ascending=[True, False])


,comparison,asset,ks_stat,ks_pvalue
19,pyscarcopula sample vs empirical data,AMGN,0.016227,0.903461
17,pyscarcopula sample vs empirical data,AMAT,0.015867,0.917059
18,pyscarcopula sample vs empirical data,AMD,0.015594,0.926604
11,pyscarcopula sample vs empirical data,ABBV,0.015066,0.943292
13,pyscarcopula sample vs empirical data,ACN,0.013158,0.983344
12,pyscarcopula sample vs empirical data,ABT,0.009423,0.999894
16,pyscarcopula sample vs empirical data,ADP,0.009416,0.999896
15,pyscarcopula sample vs empirical data,ADI,0.009140,0.999945
14,pyscarcopula sample vs empirical data,ADBE,0.008667,0.999984
10,pyscarcopula sample vs empirical data,AAPL,0.007399,1.000000


## Detailed conditional sampling comparison (`d=10`)

We fix 9 of 10 variables and compare the one remaining free coordinate. For `pyvinecopulib`, the free variable is chosen as the first variable in the fitted vine order; fixing Rosenblatt coordinates for the other variables then leaves those variables fixed exactly after `inverse_rosenblatt()`.

For `pyscarcopula`, the same conditioning values are passed as `given={...}`. If the fixed set is not an exact suffix for the fitted structure, the runtime DAG + MCMC fallback is used.

In [53]:
def timed_silent(fn):
    t0 = time.perf_counter()
    out = fn()
    return out, time.perf_counter() - t0


def pyvine_conditional_one_free(model, x0, free_var, n, seed):
    d = len(x0)
    z0 = model.rosenblatt(as_fortran(x0.reshape(1, -1)), randomize_discrete=False)
    rng = np.random.default_rng(seed)
    z = rng.uniform(1e-10, 1.0 - 1e-10, size=(n, d))
    given_vars = [j for j in range(d) if j != free_var]
    z[:, given_vars] = z0[0, given_vars]
    x = model.inverse_rosenblatt(as_fortran(z))
    return x

x0 = u10[-1].copy()
free_var = pv_order(pv_v10)[0]
given = {j: float(x0[j]) for j in range(10) if j != free_var}

pv_cond, pv_cond_t = timed_silent(
    lambda: pyvine_conditional_one_free(pv_v10, x0, free_var, N_CONDITIONAL, SEED + 100)
)

pcfg = PredictConfig(
    given=given,
    horizon='next',
    return_diagnostics=True,
    mcmc_steps=COND_MCMC_STEPS,
    mcmc_burnin=COND_MCMC_BURNIN,
)
(ps_cond, ps_cond_diag), ps_cond_t = timed_silent(
    lambda: ps_v10.predict(
        N_CONDITIONAL,
        u_train=u10,
        predict_config=pcfg,
        rng=np.random.default_rng(SEED + 101),
    )
)

fixed_vars = list(given)
pv_fixed_err = float(np.max(np.abs(pv_cond[:, fixed_vars] - x0[fixed_vars])))
ps_fixed_err = float(np.max(np.abs(ps_cond[:, fixed_vars] - x0[fixed_vars])))

ps_free = ps_cond[:, free_var]
pv_free = pv_cond[:, free_var]
ks = stats.ks_2samp(ps_free, pv_free)
cvm = stats.cramervonmises_2samp(ps_free, pv_free)

conditional_summary = pd.DataFrame([{
    'free_asset': cols10[free_var],
    'n': N_CONDITIONAL,
    'pyscarcopula_method': ps_cond_diag.get('conditional_method'),
    'pyscarcopula_time_s': ps_cond_t,
    'pyvinecopulib_time_s': pv_cond_t,
    'pyscarcopula_fixed_max_abs_error': ps_fixed_err,
    'pyvinecopulib_fixed_max_abs_error': pv_fixed_err,
    'ks_stat': ks.statistic,
    'ks_pvalue': ks.pvalue,
    'cvm_stat': cvm.statistic,
    'cvm_pvalue': cvm.pvalue,
    'wasserstein': stats.wasserstein_distance(ps_free, pv_free),
    'mean_pyscarcopula': float(np.mean(ps_free)),
    'mean_pyvinecopulib': float(np.mean(pv_free)),
    'std_pyscarcopula': float(np.std(ps_free)),
    'std_pyvinecopulib': float(np.std(pv_free)),
}])
conditional_summary.T


,0
free_asset,AMD
n,3000
pyscarcopula_method,suffix
pyscarcopula_time_s,0.029384
pyvinecopulib_time_s,0.025916
pyscarcopula_fixed_max_abs_error,0.0
pyvinecopulib_fixed_max_abs_error,0.0
ks_stat,0.018667
ks_pvalue,0.672845
cvm_stat,0.084767


In [54]:
probs = np.array([0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99])
quantiles = pd.DataFrame({
    'prob': probs,
    'pyscarcopula': np.quantile(ps_free, probs),
    'pyvinecopulib': np.quantile(pv_free, probs),
})
quantiles['diff'] = quantiles['pyscarcopula'] - quantiles['pyvinecopulib']
quantiles


,prob,pyscarcopula,pyvinecopulib,diff
0,0.01,0.016036,0.019315,-0.003279
1,0.05,0.067767,0.075335,-0.007568
2,0.10,0.132627,0.133186,-0.000559
3,0.25,0.304074,0.306574,-0.002500
4,0.50,0.572599,0.586158,-0.013558
5,0.75,0.830182,0.836642,-0.006460
6,0.90,0.933822,0.935796,-0.001974
7,0.95,0.961978,0.962830,-0.000852
8,0.99,0.985210,0.984330,0.000879


## Notes on interpreting results

- `pyvinecopulib` and `pyscarcopula` use different matrix conventions. The structure section decodes `pyvinecopulib` through `RVineStructure.order` and `struct_array(..., natural_order=False)`; raw matrix equality is not a reliable structure check, especially for truncated vines.
- For fixed-family benchmarks, `preselect_families=False` is intentional. Otherwise `pyvinecopulib` can preselect `indep` based on symmetry/tail heuristics even when `family_set` contains only one non-independent family.
- If active tree edges and pair-copula parameters match, log-likelihoods should be close. Remaining small differences usually come from pseudo-observation propagation and numerical h/inverse-h details.
- Unconditional samples are independent Monte Carlo draws. A single marginal KS p-value can be small across many assets and seeds; inspect KS effect sizes together with correlation distance and the permutation MMD p-value.
- The detailed unconditional section resamples `d=10` with a separate seed so benchmark timing seeds do not dominate the diagnostic table.
- For conditional sampling, fixing 9 variables reduces the comparison to one free coordinate. KS/CvM/Wasserstein and quantile differences are the most direct diagnostics; very large sample sizes can make p-values sensitive to tiny implementation differences.
